In [27]:
import numpy as np  
import pandas as pd
#import plotly.graph_objects as go

df=pd.read_csv('temp-files/Microplastics/whc-sites(tangibles)-2021.csv')

In [28]:
df

,Name,short_description,date_inscribed,danger,date_end,longitude,latitude,area_hectares,category_long,category_short,Country name,Region,iso_code,transboundary,rev_bis
0,L’Anse aux Meadows National Historic Site,<p>At the tip of the Great Northern Peninsula ...,1978,0,NaN,-55.616667,51.466667,7991.00,Cultural,C,Canada,Europe and North America,ca,0,NaN
1,Nahanni National Park,"<p>Located along the South Nahanni River, one ...",1978,0,NaN,-125.589444,61.547222,476560.00,Natural,N,Canada,Europe and North America,ca,0,NaN
2,Galápagos Islands,"<p>Situated in the Pacific Ocean some 1,000 km...",1978,0,2010.0,-90.501319,-0.689860,14066514.00,Natural,N,Ecuador,Latin America and the Caribbean,ec,0,Bis
3,City of Quito,"<p>Quito, the capital of Ecuador, was founded ...",1978,0,NaN,-78.512083,-0.220000,70.43,Cultural,C,Ecuador,Latin America and the Caribbean,ec,0,NaN
4,Simien National Park,<p>Massive erosion over the years on the Ethio...,1978,0,2017.0,38.066667,13.183333,13600.00,Natural,N,Ethiopia,Africa,et,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1150,The work of engineer Eladio Dieste: Church of ...,<p>The Church of Atlántida with its belfry and...,2021,0,NaN,-55.766408,-34.743919,0.56,Cultural,C,Uruguay,Latin America and the Caribbean,uy,0,NaN
1151,The Great Spa Towns of Europe,The transnational site of The Great Spa Towns ...,2021,0,NaN,5.866944,50.492222,7014.00,Cultural,C,"Austria,Belgium,Czechia,France,Germany,Italy,U...",Europe and North America,"at,be,cz,de,fr,gb,it",1,NaN
1152,Frontiers of the Roman Empire – The Danube Lim...,<p>It covers almost 600km of the whole Roman E...,2021,0,NaN,16.861389,48.115194,NaN,Cultural,C,"Austria,Germany,Slovakia",Europe and North America,"at,de,sk",1,NaN
1153,Colonies of Benevolence,<p>The transnational serial property encompass...,2021,0,NaN,6.391589,53.042222,2012.00,Cultural,C,"Belgium,Netherlands",Europe and North America,"be,nl",1,NaN


In [29]:
pos_latlon = df[['latitude','longitude']]
pos_latlon = pos_latlon.values.tolist()
pos_latlon = dict(zip(df['Name'], pos_latlon))

pos_lonlat = df[['longitude','latitude']]
pos_lonlat = pos_lonlat.values.tolist()
pos_lonlat = dict(zip(df['Name'], pos_lonlat))

In [30]:
# calculate xyz from lat lon 
import math
from math import radians, cos, sin, sqrt

def geodetic_to_geocentric(ellipsoid, latitude, longitude, height):
    """Return geocentric (Cartesian) Coordinates x, y, z corresponding to
    the geodetic coordinates given by latitude and longitude (in
    degrees) and height above ellipsoid. The ellipsoid must be
    specified by a pair (semi-major axis, reciprocal flattening).

    """
    φ = radians(latitude)
    λ = radians(longitude)
    sin_φ = sin(φ)
    a, rf = ellipsoid           # semi-major axis, reciprocal flattening
    e2 = 1 - (1 - 1 / rf) ** 2  # eccentricity squared
    n = a / sqrt(1 - e2 * sin_φ ** 2) # prime vertical radius
    r = (n + height) * cos(φ)   # perpendicular distance from z axis
    x = r * cos(λ)
    y = r * sin(λ)
    z = (n * (1 - e2) + height) * sin_φ
    return x, y, z

def cart2polar(x, y, z):
   xy = math.sqrt(x**2 + y**2) # sqrt(x² + y²) 
   x_2 = x**2
   y_2 = y**2
   z_2 = z**2
   r = math.sqrt(x_2 + y_2 + z_2) # r = sqrt(x² + y² + z²)
   theta = np.arctan2(y, x) 
   phi = np.arctan2(xy, z) 
   return r, theta, phi

def polar2cart(r, theta, phi):
    x = r * math.sin(theta) * math.cos(phi)
    y = r * math.sin(theta) * math.sin(phi)
    z = r * math.cos(theta)
    return x,y,z

In [31]:
def normalize_coordinates(x, y, z):
    """Normalize x, y, z coordinates to a range between 0 and 1."""
    magnitude = sqrt(x**2 + y**2 + z**2)
    x_norm, y_norm, z_norm = x / magnitude, y / magnitude, z / magnitude
    # Shift and scale to ensure the range is [0, 1]
    x_final = (x_norm + 1) / 2
    y_final = (y_norm + 1) / 2
    z_final = (z_norm + 1) / 2
    return x_final, y_final, z_final

In [32]:
pos_xyz = {}
for k,v in pos_latlon.items():
    x,y,z = geodetic_to_geocentric((6378136.6,298.25), v[0], v[1], 0)
    xn,yn,zn = normalize_coordinates(x, y, z)
    pos_xyz[k] = (xn,yn,zn)


# mirror coordinates along y-axis
pos_xyz_mirrored = {}
for k,v in pos_xyz.items():
    pos_xyz_mirrored[k] = (-v[0],v[1],v[2])

# scale coordinates to move slightly towards center 
scaling = 0.95
pos_xyz_scaled = {}
for k,v in pos_xyz_mirrored.items():
    pos_xyz_scaled[k] = (v[0]*scaling,v[1]*scaling,v[2]*scaling)

pos_xyz = pos_xyz_scaled

In [68]:
# color them based on cultural or natural heritage 

col_natural = '#229E00' #[150,209,0,100]
col_natural_VR = [34,158,0,110]

col_cultural = '#805602' #[220,56,4,100]
col_cultural_VR = [128,86,2,90]

col_mixed = '#F1FF33' #[241,255,51,100]
col_mix_VR = [241,255,51,80]  
 
d_nodecolors = {}
d_nodecolors_VR = {}
for k in pos_xyz.keys():
    if df[df['Name']==k]['category_long'].values[0] == 'Natural':
        d_nodecolors[k] = col_natural
        d_nodecolors_VR[k] = col_natural_VR
    elif df[df['Name']==k]['category_long'].values[0] == 'Cultural':
        d_nodecolors[k] = col_cultural
        d_nodecolors_VR[k] = col_cultural_VR
    else:   
        d_nodecolors[k] = col_mixed
        d_nodecolors_VR[k] = col_mix_VR

### annotations

In [69]:
short_desc_raw = df['short_description'].tolist()

# remove <p> tags and other html tags
import re
short_desc = []
for desc in short_desc_raw:
    desc = re.sub(r'<.*?>', '', desc)
    short_desc.append(desc)
    
short_desc[:2]

['At the tip of the Great Northern Peninsula of the island of Newfoundland, the remains of an 11th-century Viking settlement are evidence of the first European presence in North America. The excavated remains of wood-framed peat-turf buildings are similar to those found in Norse Greenland and Iceland.',
 "Located along the South Nahanni River, one of the most spectacular wild rivers in North America, this park contains deep canyons and huge waterfalls, as well as a unique limestone cave system. The park is also home to animals of the boreal forest, such as wolves, grizzly bears and caribou. Dall's sheep and mountain goats are found in the park's alpine environment."]

In [70]:
area_hectars = df['area_hectares'].tolist()

# add hectar to the end of the area
area = []
for a in area_hectars:
    area.append(str(a) + ' ha')
    

In [71]:
annotations = {}
for i in range(len(df)):
    annotations[df['Name'][i]] = {'short_desc': short_desc[i], 'category': df['category_long'][i], 'area': area[i], 'date_inscribed': df['date_inscribed'][i]}

In [72]:
selections = dict(zip(pos_xyz.keys(),df['category_long']))
selections

{'L’Anse aux Meadows National Historic Site': 'Cultural',
 'Nahanni National Park': 'Natural',
 'Galápagos Islands': 'Natural',
 'City of Quito': 'Cultural',
 'Simien National Park': 'Natural',
 'Rock-Hewn Churches, Lalibela': 'Cultural',
 'Aachen Cathedral ': 'Cultural',
 'Historic Centre of Kraków': 'Cultural',
 'Wieliczka and Bochnia Royal Salt Mines': 'Cultural',
 'Island of Gorée': 'Cultural',
 'Mesa Verde National Park': 'Cultural',
 'Yellowstone National Park': 'Natural',
 'Boyana Church': 'Cultural',
 'Madara Rider': 'Cultural',
 'Thracian Tomb of Kazanlak': 'Cultural',
 'Rock-Hewn Churches of Ivanovo': 'Cultural',
 'Dinosaur Provincial Park': 'Natural',
 'Old City of Dubrovnik': 'Cultural',
 'Historical Complex of Split with the Palace of Diocletian': 'Cultural',
 'Plitvice Lakes National Park': 'Natural',
 'Virunga National Park': 'Natural',
 'Memphis and its Necropolis – the Pyramid Fields from Giza to Dahshur': 'Cultural',
 'Ancient Thebes with its Necropolis': 'Cultural',


### make nx Graph object

In [73]:
# make nx Graph from data 

import networkx as nx

G = nx.Graph()

for k,v in pos_xyz.items():
    G.add_node(k)
    G.nodes[k]['pos'] = v
    G.nodes[k]['nodecolor'] = d_nodecolors_VR[k]
    G.nodes[k]['annotation'] = annotations[k]
    G.nodes[k]['cluster'] = selections[k] 

G.graph['projectname'] = 'Microplastics'
G.graph["layoutname"] ='01-Unesco_worldheritagesites_geo'

# create VR Project 

In [74]:
import nx2json as nx2j
nx2j.create_project(G)

Successfully created the directory static/projects/Microplastics 
PROGRESS: loaded graph JSON...
PROGRESS: stored graph data...
PROGRESS: stored layouts...
PROGRESS: stored node info...
PROGRESS: made node position textures...
PROGRESS: made textures for node colors...
PROGRESS: made textures for links...
PROGRESS: made textures for linkcolors...
PROGRESS: writing json files for project and nodes...
Project created successfully.
